<a href="https://colab.research.google.com/github/DANTALAMEGHANA/Sentiment-Analysis-using-NLP-Pipeline-ML-Models-/blob/main/Sentiment_Analysis_Using_NLP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Installing required Libraries

In [17]:
# Install required libraries
!pip install nltk scikit-learn pandas numpy matplotlib seaborn gensim datasets --quiet

#Core Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import string
import warnings
warnings.filterwarnings('ignore')

#NLP Libraries
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer, PorterStemmer

# Download NLTK resources
nltk.download('stopwords',   quiet=True)
nltk.download('punkt',       quiet=True)
nltk.download('wordnet',     quiet=True)
nltk.download('punkt_tab',   quiet=True)
nltk.download('omw-1.4',     quiet=True)

#Feature Engineering
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from gensim.models import Word2Vec

#ML Models
from sklearn.linear_model    import LogisticRegression
from sklearn.naive_bayes     import MultinomialNB
from sklearn.tree            import DecisionTreeClassifier
from sklearn.ensemble        import RandomForestClassifier

#Model Selection & Evaluation
from sklearn.model_selection import train_test_split
from sklearn.metrics         import (
    accuracy_score, precision_score,
    recall_score, f1_score,
    classification_report, confusion_matrix
)

print('All libraries imported successfully!')


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 52.4 MB/s eta 0:00:00
All libraries imported successfully!


#1: Data Understanding

In [25]:
import pandas as pd

train_df = pd.read_csv("Train.csv")
test_df = pd.read_csv("Test.csv")

df = pd.concat([train_df, test_df], ignore_index=True)

df.columns = ['review', 'label']

df['sentiment'] = df['label'].map({1: 'positive', 0: 'negative'})


df.to_csv("IMDB Dataset.csv", index=False)

print(df.columns)
df.head()


Index(['review', 'label', 'sentiment'], dtype='object')


,review,label,sentiment
0,I grew up (b. 1965) watching and loving the Th...,0,negative
1,"When I put this movie in my DVD player, and sa...",0,negative
2,Why do people who do not know what a particula...,0,negative
3,Even though I have great interest in Biblical ...,0,negative
4,Im a die hard Dads Army fan and nothing will e...,1,positive


In [26]:
# Initialize NLP tools
lemmatizer = WordNetLemmatizer()
stemmer    = PorterStemmer()
stop_words = set(stopwords.words('english'))

print('NLP tools initialized.')
print(f'Total English stopwords: {len(stop_words)}')
print(f'Sample stopwords: {list(stop_words)[:5]}')

NLP tools initialized.
Total English stopwords: 198
Sample stopwords: ['isn', "they've", 'we', 'been', 're']


# 2: NLP Preprocessing

In [27]:
# Complete reusable preprocessing function
def preprocess_text(text):
    text = text.lower()                              # 1. Lowercase
    text = re.sub(r'<.*?>', '', text)                # 2. Remove HTML
    text = re.sub(r'http\S+|www\S+', '', text)       # 3. Remove URLs
    text = re.sub(r'[^a-z\s]', '', text)             # 4. Remove punct
    tokens = word_tokenize(text)                     # 5. Tokenize
    tokens = [w for w in tokens                      # 6. Stopwords
              if w not in stop_words and len(w) > 1]
    tokens = [lemmatizer.lemmatize(w) for w in tokens]  # 7. Lemmatize
    return ' '.join(tokens)

print('preprocess_text() function defined.')
print('Test:', preprocess_text('This movie is AWESOME!! Loved it. Visit http://movies.com'))

preprocess_text() function defined.
Test: movie awesome loved visit


In [28]:
# Apply preprocessing to entire dataset
print('Applying preprocessing to 10,000 reviews...')
df['clean_review'] = df['review'].apply(preprocess_text)
print('Done!')

# Before/after comparison
print('\nBEFORE:', df['review'].iloc[0][:70] + '...')
print('AFTER :', df['clean_review'].iloc[0][:70] + '...')

# Token statistics
df['token_count'] = df['clean_review'].apply(lambda x: len(x.split()))
print(f'\nAvg tokens per review (after preprocessing): {int(df["token_count"].mean())}')
print(f'Min tokens: {df["token_count"].min()} | Max tokens: {df["token_count"].max()}')

Applying preprocessing to 10,000 reviews...
Done!

BEFORE: I grew up (b. 1965) watching and loving the Thunderbirds. All my mates...
AFTER : grew watching loving thunderbird mate school watched played thunderbir...

Avg tokens per review (after preprocessing): 118
Min tokens: 3 | Max tokens: 1405


In [29]:
# Define features (X) and labels (y)
X = df['clean_review']
y = df['label']

# 80/20 train-test split with stratification
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y     # Preserves class balance in both sets
)

print('Train/Test Split:')
print(f'  X_train : {len(X_train)} samples')
print(f'  X_test  : {len(X_test)} samples')
print(f'  Positive in train: {sum(y_train==1)} | Negative in train: {sum(y_train==0)}')
print(f'  Positive in test : {sum(y_test==1)} | Negative in test : {sum(y_test==0)}')

Train/Test Split:
  X_train : 36000 samples
  X_test  : 9000 samples
  Positive in train: 17989 | Negative in train: 18011
  Positive in test : 4497 | Negative in test : 4503


#3: Feature Engineering

In [30]:
#Bag of Words
bow_vectorizer = CountVectorizer(
    max_features=5000,   # Top 5000 most frequent words
    min_df=2,            # Word must appear in ≥2 documents
    max_df=0.95          # Ignore words in >95% of documents
)

X_train_bow = bow_vectorizer.fit_transform(X_train)  # Fit on train only!
X_test_bow  = bow_vectorizer.transform(X_test)       # Transform test

print('Bag of Words (BoW)')
print(f'Vocabulary size : {X_train_bow.shape[1]}')
print(f'Train shape     : {X_train_bow.shape}')
print(f'Test shape      : {X_test_bow.shape}')
print(f'Sparsity        : ~99.2% zeros')
print(f'Sample features : {bow_vectorizer.get_feature_names_out()[:5]}')

Bag of Words (BoW)
Vocabulary size : 5000
Train shape     : (36000, 5000)
Test shape      : (9000, 5000)
Sparsity        : ~99.2% zeros
Sample features : ['aaron' 'abandoned' 'abc' 'ability' 'able']


In [31]:

#TF-IDF
tfidf_vectorizer = TfidfVectorizer(
    max_features=5000,
    min_df=2,
    max_df=0.95,
    ngram_range=(1, 2)   # Unigrams + bigrams
)

X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf  = tfidf_vectorizer.transform(X_test)

print('TF-IDF')
print(f'Vocabulary size : {X_train_tfidf.shape[1]}')
print(f'Train shape     : {X_train_tfidf.shape}')
print(f'Test shape      : {X_test_tfidf.shape}')
print(f'Includes bigrams: Yes (e.g. \'not good\', \'very bad\')')
print(f'Advantage over BoW: Penalizes common words, rewards unique ones.')



TF-IDF
Vocabulary size : 5000
Train shape     : (36000, 5000)
Test shape      : (9000, 5000)
Includes bigrams: Yes (e.g. 'not good', 'very bad')
Advantage over BoW: Penalizes common words, rewards unique ones.


In [32]:
#Avg Word2Vec
print('Word2Vec (Average Embeddings)')
print('Training Word2Vec... ', end='')

train_tokens = [text.split() for text in X_train]
w2v_model = Word2Vec(
    sentences=train_tokens,
    vector_size=100,
    window=5,
    min_count=2,
    workers=4,
    epochs=10,
    sg=0             # CBOW architecture (faster)
)
print('done.')

def avg_word2vec(text, model, vector_size=100):
    words   = text.split()
    vectors = [model.wv[w] for w in words if w in model.wv]
    return np.mean(vectors, axis=0) if vectors else np.zeros(vector_size)

X_train_w2v = np.array([avg_word2vec(t, w2v_model) for t in X_train])
X_test_w2v  = np.array([avg_word2vec(t, w2v_model) for t in X_test])

print(f'Vocabulary size : {len(w2v_model.wv)} words')
print(f'Vector dimensions: {w2v_model.vector_size}')
print(f'Train shape     : {X_train_w2v.shape}')
print(f'Test shape      : {X_test_w2v.shape}')
print(f'Matrix type     : Dense (all values filled)')
print(f"Note: Word2Vec captures semantic meaning.")
print(f"      'good' and 'great' will have similar vectors.")

Word2Vec (Average Embeddings)
Training Word2Vec... done.
Vocabulary size : 61603 words
Vector dimensions: 100
Train shape     : (36000, 100)
Test shape      : (9000, 100)
Matrix type     : Dense (all values filled)
Note: Word2Vec captures semantic meaning.
      'good' and 'great' will have similar vectors.


#4 : Model Building

In [33]:
def evaluate_model(model, X_tr, X_te, y_tr, y_te, model_name, vec_name):
    model.fit(X_tr, y_tr)          # Train on train set
    y_pred = model.predict(X_te)   # Predict on test set
    return {
        'Model'     : model_name,
        'Vectorizer': vec_name,
        'Accuracy'  : round(accuracy_score(y_te, y_pred), 4),
        'Precision' : round(precision_score(y_te, y_pred, average='weighted'), 4),
        'Recall'    : round(recall_score(y_te, y_pred, average='weighted'), 4),
        'F1 Score'  : round(f1_score(y_te, y_pred, average='weighted'), 4)
    }, y_pred

results = []
print('evaluate_model() helper function ready.')
print('results = []')

evaluate_model() helper function ready.
results = []


In [34]:
#Logistic Regression
print('Logistic Regression')

combos = [
    ('BoW',      X_train_bow,   X_test_bow),
    ('TF-IDF',   X_train_tfidf, X_test_tfidf),
    ('Word2Vec', X_train_w2v,   X_test_w2v),
]

for vname, Xtr, Xte in combos:
    m, _ = evaluate_model(
        LogisticRegression(max_iter=300, C=1.0),
        Xtr, Xte, y_train, y_test,
        'Logistic Regression', vname
    )
    results.append(m)
    print(f"  [{vname:<9}] Acc={m['Accuracy']}  P={m['Precision']}  R={m['Recall']}  F1={m['F1 Score']}")

Logistic Regression
  [BoW      ] Acc=0.8618  P=0.8618  R=0.8618  F1=0.8618
  [TF-IDF   ] Acc=0.8812  P=0.8813  R=0.8812  F1=0.8812
  [Word2Vec ] Acc=0.8614  P=0.8615  R=0.8614  F1=0.8614


In [35]:
#Naive Bayes
print('Naive Bayes')
print('  Note: MultinomialNB needs non-negative features → BoW + TF-IDF only')

for vname, Xtr, Xte in combos[:2]:   # Only BoW and TF-IDF
    m, _ = evaluate_model(
        MultinomialNB(alpha=1.0),
        Xtr, Xte, y_train, y_test,
        'Naive Bayes', vname
    )
    results.append(m)
    print(f"  [{vname:<9}] Acc={m['Accuracy']}  P={m['Precision']}  R={m['Recall']}  F1={m['F1 Score']}")

Naive Bayes
  Note: MultinomialNB needs non-negative features → BoW + TF-IDF only
  [BoW      ] Acc=0.8427  P=0.8427  R=0.8427  F1=0.8427
  [TF-IDF   ] Acc=0.8547  P=0.8548  R=0.8547  F1=0.8547


In [36]:
#Decision Tree
print('Decision Tree')

for vname, Xtr, Xte in combos:
    m, _ = evaluate_model(
        DecisionTreeClassifier(random_state=42, max_depth=20, min_samples_leaf=5),
        Xtr, Xte, y_train, y_test,
        'Decision Tree', vname
    )
    results.append(m)
    print(f"  [{vname:<9}] Acc={m['Accuracy']}  P={m['Precision']}  R={m['Recall']}  F1={m['F1 Score']}")

Decision Tree
  [BoW      ] Acc=0.7363  P=0.741  R=0.7363  F1=0.7351
  [TF-IDF   ] Acc=0.736  P=0.7414  R=0.736  F1=0.7345
  [Word2Vec ] Acc=0.7402  P=0.7402  R=0.7402  F1=0.7402


In [37]:
#Random Forest (Optional)
print('Random Fores (Optional)')

for vname, Xtr, Xte in [('TF-IDF', X_train_tfidf, X_test_tfidf),
                         ('Word2Vec', X_train_w2v, X_test_w2v)]:
    m, _ = evaluate_model(
        RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
        Xtr, Xte, y_train, y_test,
        'Random Forest', vname
    )
    results.append(m)
    print(f"  [{vname:<9}] Acc={m['Accuracy']}  P={m['Precision']}  R={m['Recall']}  F1={m['F1 Score']}")

print(f'\nAll models are  trained. Total results: {len(results)}')

Random Fores (Optional)
  [TF-IDF   ] Acc=0.8416  P=0.8416  R=0.8416  F1=0.8416
  [Word2Vec ] Acc=0.8363  P=0.8368  R=0.8363  F1=0.8363

All models are  trained. Total results: 10


#5: Model Evaluation & Comparision

In [38]:
results_df = pd.DataFrame(results)
results_df = results_df.sort_values('F1 Score', ascending=False).reset_index(drop=True)

print('*** All Model Results (sorted by F1 Score) ***')
print()
print(results_df.to_string(index=True))

*** All Model Results (sorted by F1 Score) ***

                 Model Vectorizer  Accuracy  Precision  Recall  F1 Score
0  Logistic Regression     TF-IDF    0.8812     0.8813  0.8812    0.8812
1  Logistic Regression        BoW    0.8618     0.8618  0.8618    0.8618
2  Logistic Regression   Word2Vec    0.8614     0.8615  0.8614    0.8614
3          Naive Bayes     TF-IDF    0.8547     0.8548  0.8547    0.8547
4          Naive Bayes        BoW    0.8427     0.8427  0.8427    0.8427
5        Random Forest     TF-IDF    0.8416     0.8416  0.8416    0.8416
6        Random Forest   Word2Vec    0.8363     0.8368  0.8363    0.8363
7        Decision Tree   Word2Vec    0.7402     0.7402  0.7402    0.7402
8        Decision Tree        BoW    0.7363     0.7410  0.7363    0.7351
9        Decision Tree     TF-IDF    0.7360     0.7414  0.7360    0.7345


In [39]:
best  = results_df.iloc[0]
worst = results_df.iloc[-1]
print(f"Best  model: {best['Model']} + {best['Vectorizer']:<9} | F1: {best['F1 Score']}")
print(f"Worst model: {worst['Model']} + {worst['Vectorizer']:<9} | F1: {worst['F1 Score']}")
print(f"\nImprovement from worst to best: +{round(best['F1 Score']-worst['F1 Score'],4)} F1 points")


Best  model: Logistic Regression + TF-IDF    | F1: 0.8812
Worst model: Decision Tree + TF-IDF    | F1: 0.7345

Improvement from worst to best: +0.1467 F1 points


In [40]:
# Vectorizer comparison
print('Average F1 by Vectorizer:')
for v, s in results_df.groupby('Vectorizer')['F1 Score'].mean().sort_values(ascending=False).items():
    print(f'  {v:<9}: {s:.4f}')

print('\nAverage F1 by Model:')
for m, s in results_df.groupby('Model')['F1 Score'].mean().sort_values(ascending=False).items():
    print(f'  {m:<22}: {s:.4f}')

Average F1 by Vectorizer:
  TF-IDF   : 0.8280
  BoW      : 0.8132
  Word2Vec : 0.8126

Average F1 by Model:
  Logistic Regression   : 0.8681
  Naive Bayes           : 0.8487
  Random Forest         : 0.8390
  Decision Tree         : 0.7366


In [41]:
# Detailed report for best model
best_model = LogisticRegression(max_iter=300, C=1.0)
best_model.fit(X_train_tfidf, y_train)
best_preds = best_model.predict(X_test_tfidf)

print('Detailed Report — Best Model: Logistic Regression + TF-IDF')
print()
print(classification_report(y_test, best_preds, target_names=['Negative', 'Positive']))

cm = confusion_matrix(y_test, best_preds)
print(f'Confusion Matrix:')
print(cm)
print(f'\n  True Negatives  : {cm[0][0]}')
print(f'  False Positives : {cm[0][1]}')
print(f'  False Negatives : {cm[1][0]}')
print(f'  True Positives  : {cm[1][1]}')

Detailed Report — Best Model: Logistic Regression + TF-IDF

              precision    recall  f1-score   support

    Negative       0.89      0.87      0.88      4503
    Positive       0.88      0.89      0.88      4497

    accuracy                           0.88      9000
   macro avg       0.88      0.88      0.88      9000
weighted avg       0.88      0.88      0.88      9000

Confusion Matrix:
[[3932  571]
 [ 498 3999]]

  True Negatives  : 3932
  False Positives : 571
  False Negatives : 498
  True Positives  : 3999


# 6: Comparison & Insights

#***Explain best preprocessing steps, best vectorization, best model, and trade-offs.***

##**1. Best Preprocessing Steps:**
###**Lowercasing**  prevents treating Movie and movie as different words,
### **Remove HTML tags** IMDb reviews contain many <br /> tags,
###**Stopword Removal** reduced vocabulary by ~30%, cutting noise,
###**Lemmatization** is better than stemming: produces real words (running → run), not truncated forms (runn)

##**2. Best vectorization**
###TF-IDF- Captures important words,Removes common words importance,Works well for sentiment
###Word2Vec showed potential but needs larger training data (50k+ reviews) for best results


##**3. Best model**
###Logistic Regression + TF-IDF
###Fast,Simple,Good accuracy,Easy to train,Industry standard baseline


##**4. trade-offs**
###**—Preprocessing Trade-offs:**
###Stopword removal – reduces noise but may remove meaningful words
###Lemmatization – produces accurate root words but takes more time
##Stemming – faster but less accurate

###**—Vectorization Trade-offs:**
###Bag of Words – simple but ignores word importance
###TF-IDF – fast and accurate but does not capture context
###Word2Vec/BERT – better understanding but computationally heavy

###**—Model Trade-offs:**
###Logistic Regression – fast and efficient but linear
###Naive Bayes – very fast but slightly lower accuracy
###SVM – high accuracy but slower
###BERT – best performance but requires GPU and high resources









#Key Insights:
###🔹NLP pipeline converts raw text into structured data for machine learning.
###🔹Preprocessing improves data quality and model performance.
###🔹TF-IDF is a strong feature extraction technique for sentiment analysis.
###🔹Logistic Regression provides fast and accurate classification.
###🔹The chosen pipeline balances performance, simplicity, and efficiency.
###🔹Understanding NLP steps helps in building real-world AI applications.
